# ComfyUI on Colab — Personalised Launcher

**Cell 1** — setup + pipeline selector. Run it, pick pipelines in the widget, click **Save selection**.

**Cell 2** — installs deps, downloads weights, launches ComfyUI and opens a public cloudflared URL.

**Cell 3** — quick restart. Kills the running ComfyUI and relaunches it without re-downloading anything. The public URL stays the same.

In [ ]:
# ====================================================================
# CELL 1 — SETUP + PIPELINE SELECTOR
# Run this cell, wait for the widget, pick pipelines + LoRAs,
# then click "Save selection" before running Cell 2.
# ====================================================================

import os, subprocess, sys

PREFS_PATH = "/content/.comfyui-colab-prefs.json"

# --- Tokens (Colab userdata, env fallback, then prompt) -------------
# Add HF_TOKEN and CIVITAI_TOKEN in the key-icon sidebar in Colab and
# toggle "Notebook access" on. Local runs use $HF_TOKEN / $CIVITAI_TOKEN.
def _load_token(name, prompt_if_missing=True):
    if os.environ.get(name):
        return
    try:
        from google.colab import userdata  # type: ignore
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            return
    except Exception:
        pass
    if prompt_if_missing:
        import getpass
        os.environ[name] = getpass.getpass(f"{name} (leave blank to skip): ") or ""

_load_token("HF_TOKEN")
_load_token("CIVITAI_TOKEN")

# --- GPU detect + conditional torch reinstall ------------------------
def _cc():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
            text=True).strip().splitlines()[0]
        name, cc = [s.strip() for s in out.split(",", 1)]
        maj, mn = cc.split(".")
        return name, int(maj) * 10 + int(mn)
    except Exception:
        return "CPU", 0

GPU_NAME, GPU_CC = _cc()
print(f"GPU: {GPU_NAME} (compute_cap={GPU_CC})")

import torch
arches = torch.cuda.get_arch_list() if torch.cuda.is_available() else []
max_arch = max((int(a.split("_")[-1]) for a in arches if a.startswith("sm_")), default=0)
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()} max_arch=sm_{max_arch}")
if GPU_CC >= 120 and max_arch < 120:
    print("Reinstalling torch for cu128…")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "torch", "torchvision", "torchaudio",
        "--index-url", "https://download.pytorch.org/whl/cu128"])
    print("Reinstalled. RESTART runtime, then re-run this cell.")
    raise SystemExit(0)

# --- Deps ------------------------------------------------------------
subprocess.check_call(["apt-get", "install", "-y", "-qq", "aria2"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "ipywidgets", "huggingface_hub", "huggingface_hub[cli]"])

# --- Clone repo ------------------------------------------------------
REPO_URL = os.environ.get("COMFYUI_COLAB_REPO_URL",
                          "https://github.com/SamuelD27/ComfyCustom.git")
REPO_BRANCH = os.environ.get("COMFYUI_COLAB_BRANCH", "Collab")
REPO_DIR = "/content/ComfyUI"
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth", "1",
                           "--branch", REPO_BRANCH, REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"Repo ready at {REPO_DIR} (branch: {REPO_BRANCH})")

# --- Selector UI -----------------------------------------------------
import ipywidgets as W
from IPython.display import display
from colab.launcher_helpers import (
    load_registry, list_pipelines, pipeline_loras, load_prefs, save_prefs,
)

REGISTRY_PATH = f"{REPO_DIR}/scripts/model_registry.json"
registry = load_registry(REGISTRY_PATH)
pipelines = list_pipelines(registry)
prefs = load_prefs(PREFS_PATH)
sel_prev = set(prefs.get("pipelines", []))
lora_prev = {k: set(v) for k, v in prefs.get("loras", {}).items()}

STATE = {"pipelines": set(), "loras": {}, "prefs_path": PREFS_PATH,
         "registry": registry, "repo_dir": REPO_DIR, "ready": False}

pipeline_boxes = []
for p in pipelines:
    cb = W.Checkbox(
        value=(p["key"] in sel_prev),
        description=f"{p['display_name']}  ({p['total_size_gb']} GB, "
                    f"{p['model_count']} models, {p['lora_count']} LoRAs)",
        indent=False, layout=W.Layout(width="100%"))
    pipeline_boxes.append((p["key"], cb))

lora_pane = W.VBox([], layout=W.Layout(width="100%"))

def _update_lora(*_):
    STATE["pipelines"] = {k for k, cb in pipeline_boxes if cb.value}
    kids, STATE["loras"] = [], {}
    for key in sorted(STATE["pipelines"]):
        loras = pipeline_loras(registry, key)
        if not loras:
            continue
        opts = [(f"{l['display_name']} ({l.get('size_gb', 0):.2f} GB)", l["filename"])
                for l in loras]
        pre = [fn for _d, fn in opts if fn in lora_prev.get(key, set())]
        sm = W.SelectMultiple(options=opts, value=tuple(pre), description=key,
                              rows=min(6, len(opts)),
                              layout=W.Layout(width="100%"))
        STATE["loras"][key] = set(pre)
        def _h(change, _k=key): STATE["loras"][_k] = set(change["new"])
        sm.observe(_h, names="value")
        kids.append(sm)
    lora_pane.children = kids

for _, cb in pipeline_boxes:
    cb.observe(_update_lora, names="value")

save_btn = W.Button(description="Save selection",
                    button_style="success", icon="check")
status = W.HTML("<i>Pick pipelines + LoRAs, then click <b>Save selection</b>. "
                "Then run Cell 2.</i>")

def _save(_b):
    if not STATE["pipelines"]:
        status.value = "<span style='color:#a00'>Select at least one pipeline.</span>"
        return
    save_prefs(PREFS_PATH, {
        "pipelines": sorted(STATE["pipelines"]),
        "loras": {k: sorted(v) for k, v in STATE["loras"].items()},
    })
    STATE["ready"] = True
    n_loras = sum(len(v) for v in STATE["loras"].values())
    status.value = (f"<b>Saved.</b> {len(STATE['pipelines'])} pipelines, "
                    f"{n_loras} LoRAs. Now run Cell 2.")

save_btn.on_click(_save)
_update_lora()
display(W.VBox([
    W.HTML("<h3>Pipelines</h3>"),
    W.VBox([cb for _, cb in pipeline_boxes]),
    W.HTML("<h3>LoRAs (per selected pipeline)</h3>"),
    lora_pane,
    save_btn, status,
]))


In [ ]:
# ====================================================================
# CELL 2 -- INSTALL DEPS (uv pip) + DOWNLOAD WEIGHTS + LAUNCH COMFYUI
# Run this AFTER clicking "Save selection" in Cell 1.
# ====================================================================

import os, subprocess, sys, socket, tempfile, threading, time, urllib.request
from pathlib import Path
from colab.launcher_helpers import (
    pipeline_models, pipeline_loras,
    build_hf_download_cmd, build_aria2c_cmd, write_auth_header,
    extract_trycloudflare_url,
)

if not STATE.get("ready"):
    raise SystemExit("Click 'Save selection' in Cell 1 first.")

REPO_DIR = STATE["repo_dir"]
registry = STATE["registry"]
MODELS_ROOT = Path(REPO_DIR) / "models"

# --- Ultra-fast pip via uv + hf_transfer -----------------------------
# uv: parallel resolver/downloader; hf_transfer: multi-connection HF downloads.
os.environ["UV_CONCURRENT_DOWNLOADS"] = "50"
os.environ["UV_HTTP_TIMEOUT"] = "120"
os.environ["UV_NO_PROGRESS"] = "1"
os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "--upgrade", "uv", "hf_transfer"])

UV_PIP = ["uv", "pip", "install", "--system", "--quiet",
          "--index-strategy", "unsafe-best-match"]

print("Installing ComfyUI requirements (uv pip)...")
subprocess.check_call(UV_PIP + ["-r", f"{REPO_DIR}/requirements.txt"])
for cn in sorted(Path(f"{REPO_DIR}/custom_nodes").iterdir()):
    if cn.name.endswith(".disabled") or not cn.is_dir():
        continue
    req = cn / "requirements.txt"
    if req.exists():
        print(f"  custom_nodes/{cn.name}/requirements.txt")
        subprocess.check_call(UV_PIP + ["-r", str(req)])

# --- Ultra-fast HF / aria2c download env -----------------------------
cfg = registry.get("config", {}) if isinstance(registry, dict) else {}
for k, v in cfg.get("hf_env", {}).items():
    os.environ.setdefault(k, str(v))
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

HF_MAX_WORKERS = int(os.environ.get("HF_MAX_WORKERS", "64"))
ARIA_CONNECTIONS = int(os.environ.get("ARIA_CONNECTIONS", "16"))

AUTH_DIR = Path(tempfile.mkdtemp(prefix="colab_auth_"))
auth_hdr = write_auth_header(os.environ.get("CIVITAI_TOKEN"), AUTH_DIR)

def _download_spec(spec, dest_dir):
    """Download an HF- or CivitAI-sourced model/LoRA spec to dest_dir."""
    dest_dir.mkdir(parents=True, exist_ok=True)
    target = dest_dir / spec["filename"]
    if target.exists():
        print(f"  skip (exists): {spec['filename']}")
        return
    src = spec.get("source", "hf")
    size = spec.get("size_gb", 0) or 0
    if src == "hf":
        print(f"  HF: {spec['filename']} ({size:.2f} GB)")
        cmd = build_hf_download_cmd(
            spec["hf_repo"], spec["hf_file"], str(dest_dir),
            max_workers=HF_MAX_WORKERS,
        )
        subprocess.check_call(cmd, env={**os.environ})
        remote = dest_dir / spec["hf_file"]
        if remote != target and remote.exists():
            remote.rename(target)
            # Clean up any now-empty nested dirs under dest_dir.
            p = remote.parent
            while p != dest_dir and p.is_dir() and not any(p.iterdir()):
                p.rmdir()
                p = p.parent
    elif src == "civitai":
        print(f"  CivitAI: {spec['filename']} ({size:.2f} GB)")
        cmd = build_aria2c_cmd(
            spec["civitai_url"], str(dest_dir), spec["filename"],
            auth_header_file=str(auth_hdr) if auth_hdr else None,
            connections=ARIA_CONNECTIONS,
        )
        subprocess.check_call(cmd)
    else:
        raise ValueError(f"Unknown source for {spec.get('filename')}: {src}")

for pkey in sorted(STATE["pipelines"]):
    print(f"\n=== {pkey} ===")
    for m in pipeline_models(registry, pkey):
        _download_spec(m, MODELS_ROOT / m["dest_subdir"])
    for l in pipeline_loras(registry, pkey):
        if l["filename"] not in STATE["loras"].get(pkey, set()):
            continue
        _download_spec(l, MODELS_ROOT / "loras")

# --- Sync workflows into user/default/workflows ----------------------
# Workflows live under colab/workflows/ in the repo (gitignored at
# user/default/workflows/). Copy them into the live location so ComfyUI
# picks them up on start.
import shutil
WF_SRC = Path(REPO_DIR) / "colab" / "workflows"
WF_DST = Path(REPO_DIR) / "user" / "default" / "workflows"
if WF_SRC.is_dir():
    WF_DST.mkdir(parents=True, exist_ok=True)
    n = 0
    for s in WF_SRC.rglob("*.json"):
        rel = s.relative_to(WF_SRC)
        d = WF_DST / rel
        d.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(s, d)
        n += 1
    print(f"Workflows synced: {n} files → {WF_DST}")
else:
    print(f"No workflows dir found at {WF_SRC}")

# --- Launch ComfyUI --------------------------------------------------
LOG = "/content/comfyui.log"
comfy_proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd=REPO_DIR,
    stdout=open(LOG, "a"),
    stderr=subprocess.STDOUT,
)
print(f"\nComfyUI started (pid={comfy_proc.pid}). Logs: {LOG}")

# --- Cloudflared tunnel ----------------------------------------------
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("Downloading cloudflared...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64",
        "/usr/local/bin/cloudflared")
    os.chmod("/usr/local/bin/cloudflared", 0o755)

PORT = 8188
PUBLIC_URL = {"url": None}

def _tunnel():
    for _ in range(300):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(("127.0.0.1", PORT)) == 0:
                break
        time.sleep(1)
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1)
    for line in proc.stdout:
        if PUBLIC_URL["url"] is None:
            url = extract_trycloudflare_url(line)
            if url:
                PUBLIC_URL["url"] = url
                print(f"\n*** PUBLIC URL: {url} ***\n")

threading.Thread(target=_tunnel, daemon=True).start()

print("Waiting for public URL...")
for _ in range(300):
    if PUBLIC_URL["url"]:
        break
    time.sleep(1)
print(f"Final URL: {PUBLIC_URL['url']}")

In [ ]:
# ====================================================================
# CELL 3 — RESTART COMFYUI (no re-download, tunnel preserved)
# Use this after editing workflows, tweaking configs, or when ComfyUI
# crashed. Models, deps, and the cloudflared tunnel are reused.
# ====================================================================

import os, signal, subprocess, sys, time, socket
from pathlib import Path

REPO_DIR = globals().get("REPO_DIR", "/content/ComfyUI")
LOG = globals().get("LOG", "/content/comfyui.log")
PORT = globals().get("PORT", 8188)

# --- Stop any running ComfyUI ---------------------------------------
existing = globals().get("comfy_proc")
if existing is not None and existing.poll() is None:
    print(f"Stopping ComfyUI (pid={existing.pid})...")
    existing.terminate()
    try:
        existing.wait(timeout=15)
    except subprocess.TimeoutExpired:
        existing.kill()
        existing.wait()
else:
    # Fallback: kill any stray main.py
    try:
        subprocess.run(["pkill", "-f", f"python.*{REPO_DIR}/main.py"], check=False)
    except Exception:
        pass
    time.sleep(1)

# --- Optional: pull latest repo + resync workflows ------------------
RESTART_PULL = bool(int(os.environ.get("COMFYUI_RESTART_PULL", "1")))
if RESTART_PULL and os.path.isdir(os.path.join(REPO_DIR, ".git")):
    try:
        subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])
    except subprocess.CalledProcessError as e:
        print(f"git pull skipped: {e}")

import shutil
WF_SRC = Path(REPO_DIR) / "colab" / "workflows"
WF_DST = Path(REPO_DIR) / "user" / "default" / "workflows"
if WF_SRC.is_dir():
    WF_DST.mkdir(parents=True, exist_ok=True)
    n = 0
    for s in WF_SRC.rglob("*.json"):
        rel = s.relative_to(WF_SRC)
        d = WF_DST / rel
        d.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(s, d)
        n += 1
    print(f"Workflows resynced: {n} files")

# --- Relaunch --------------------------------------------------------
comfy_proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "0.0.0.0", "--port", str(PORT)],
    cwd=REPO_DIR,
    stdout=open(LOG, "a"),
    stderr=subprocess.STDOUT,
)
print(f"ComfyUI restarted (pid={comfy_proc.pid}). Logs: {LOG}")

# --- Wait until it is listening again -------------------------------
for _ in range(180):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(("127.0.0.1", PORT)) == 0:
            break
    time.sleep(1)
else:
    print("WARNING: ComfyUI did not start listening within 3 min. Check logs:")
    print(f"  !tail -n 80 {LOG}")

url = (globals().get("PUBLIC_URL") or {}).get("url")
if url:
    print(f"Public URL (unchanged): {url}")
else:
    print("Public URL not set in this kernel (Cell 2 was not run). "
          "Re-run Cell 2 to create a new tunnel.")
